In [ ]:
# roller_coaster_detection.py
import cv2
import numpy as np
import pandas as pd
from datetime import datetime
import torch
from facenet_pytorch import MTCNN
import tensorflow as tf

# Load your single Keras model
model = tf.keras.models.load_model('Age_Sex_Detection.keras')

# Initialize MTCNN for face detection
mtcnn = MTCNN(keep_all=True, device='cuda' if torch.cuda.is_available() else 'cpu')

# Initialize data storage
data_log = []
output_file = 'roller_coaster_log.csv'  # Change to 'roller_coaster_log.xlsx' for Excel

# Preprocess face for your model
def preprocess_face(face):
    face = cv2.resize(face, (128, 128))  # Adjust to your model's input size
    face = face / 255.0  # Normalize (modify if your model uses [-1,1])
    face = np.expand_dims(face, axis=0)  # Add batch dimension
    return face

# Initialize webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Could not read frame.")
            break

        # Detect faces
        boxes, _ = mtcnn.detect(frame)

        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = map(int, box)
                face = frame[y1:y2, x1:x2]

                if face.size == 0:
                    continue

                # Preprocess face
                processed_face = preprocess_face(face)

                # Predict age and gender
                predictions = model.predict(processed_face, verbose=0)[0]  # Single model output
                age = int(predictions[0])  # Assuming first output is age (regression)
                gender_prob = predictions[1]  # Assuming second output is gender probability
                gender = 'Male' if gender_prob > 0.5 else 'Female'

                
                # Apply roller coaster restrictions
                if age < 13 or age > 60:
                    color = (0, 0, 255)  # Red
                    label = f'Not allowed (Age: {age}, {gender})'
                else:
                    color = (0, 255, 0)  # Green
                    label = f'Allowed (Age: {age}, {gender})'

                # Draw rectangle and label
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

                # Log data
                entry_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                data_log.append({
                    'Entry_Time': entry_time,
                    'Age': age,
                    'Gender': gender,
                    'Allowed': 'No' if age < 13 or age > 60 else 'Yes'
                })

        # Display frame
        cv2.imshow('Horror Roller Coaster Detection', frame)

        # Save data periodically (every 10 detections)
        if len(data_log) % 10 == 0 and len(data_log) > 0:
            df_log = pd.DataFrame(data_log)
            df_log.to_csv(output_file, index=False)
            # For Excel, uncomment below
            # df_log.to_excel(output_file, index=False)
            print(f"Saved {len(data_log)} entries to {output_file}")

        # Exit on 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except KeyboardInterrupt:
    print("Stopped by user")

finally:
    # Release resources
    cap.release()
    cv2.destroyAllWindows()

    # Final save
    if len(data_log) > 0:
        df_log = pd.DataFrame(data_log)
        df_log.to_csv(output_file, index=False)
        # For Excel, uncomment below
        # df_log.to_excel(output_file, index=False)
        print(f"Final save: {len(data_log)} entries to {output_file}")